# SPWM Colab Offload: ANN-Rate Seeds 101 & 2024 + Table 1 Open-Loop Rollout

This notebook runs the GPU-heavy tasks offloaded from local macOS:
1. **Part 1:** `ANN-Rate` (`train_representation` 160 epochs + `train_m1` 240 epochs) for seeds `101` and `2024` on `--device cuda`, with Google Drive-backed checkpoints (`--resume` on reconnect).
2. **Part 2:** `scripts/eval_table1_ssot.py` ($N=400, H=100, t_{\text{start}}=4$).

**Before running:**
- Select a GPU runtime (`Runtime -> Change runtime type -> T4 / L4 / A100 GPU`).
- Push your local working branch to GitHub.
- Upload `sim/ExportedTrajectories/` (1.9 GB) to Google Drive (e.g., `MyDrive/spwm_offload/ExportedTrajectories`).

In [ ]:
# 1. Mount Google Drive & Configure Paths
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = "https://github.com/Mattia-Maiorano/Spiking-Neural_Novelty.git"  # Use SSH/HTTPS or PAT if private
BRANCH = "wip/offload-evals"  # Update to your pushed branch name
DRIVE_ROOT = "/content/drive/MyDrive/spwm_offload"
DRIVE_DATA_DIR = f"{DRIVE_ROOT}/ExportedTrajectories"
DRIVE_RESULTS_DIR = f"{DRIVE_ROOT}/results"
WORKDIR = "/content/Spiking-Neural_Novelty"

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
!nvidia-smi

In [ ]:
# 2. Clone / Pull Repo & Install Dependencies
import os
if not os.path.exists(WORKDIR):
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git checkout {BRANCH} && git pull origin {BRANCH}
!pip install -q --no-deps -e .
!pip install -q chronicle-py pyyaml tensorboard matplotlib pandas scipy

In [ ]:
# 3. Stage 1.9 GB ExportedTrajectories to Local Colab SSD for Fast I/O
import os, shutil
LOCAL_DATA_DIR = f"{WORKDIR}/sim/ExportedTrajectories"
DRIVE_DATA_TAR = f"{DRIVE_ROOT}/ExportedTrajectories.tar"
if not os.path.exists(LOCAL_DATA_DIR) or len(os.listdir(LOCAL_DATA_DIR)) == 0:
    os.makedirs(f"{WORKDIR}/sim", exist_ok=True)
    if os.path.exists(DRIVE_DATA_TAR):
        print(f"Extracting {DRIVE_DATA_TAR} -> {WORKDIR}/sim/ ...")
        !tar -xf "{DRIVE_DATA_TAR}" -C "{WORKDIR}/sim"
    else:
        print(f"Copying {DRIVE_DATA_DIR} -> {LOCAL_DATA_DIR} ...")
        !cp -r "{DRIVE_DATA_DIR}" "{LOCAL_DATA_DIR}"
else:
    print("ExportedTrajectories already staged locally.")
!ls -1 {LOCAL_DATA_DIR} | wc -l

In [ ]:
# 4. Part 1: Train ANN-Rate Seeds 101 & 2024 on CUDA (Crash-Safe via Drive)
import os, subprocess

def run_cmd(cmd):
    print("\n>>>", " ".join(cmd))
    subprocess.run(cmd, check=True)

for s in [101, 2024]:
    rep_dir = f"{DRIVE_RESULTS_DIR}/ann_ablation/seed_{s}_representation"
    m1_dir = f"{DRIVE_RESULTS_DIR}/ann_ablation/seed_{s}_m1"
    os.makedirs(rep_dir, exist_ok=True)
    os.makedirs(m1_dir, exist_ok=True)

    rep_model = f"{rep_dir}/model.pt"
    rep_ckpt = f"{rep_dir}/checkpoint.pt"
    if not os.path.exists(rep_model):
        cmd = [
            "python", "experiments/train.py",
            "--config", "results/spwm_v0_12_4_train_representation/config.yaml",
            "--model-type", "ann_rate",
            "--mode", "train_representation",
            "--seed", str(s),
            "--epochs", "160",
            "--device", "cuda",
            "--output-dir", rep_dir,
        ]
        if os.path.exists(rep_ckpt):
            cmd.extend(["--resume", rep_ckpt])
        else:
            cmd.append("--from-scratch")
        run_cmd(cmd)
    else:
        print(f"Seed {s} representation already complete: {rep_model}")

    m1_model = f"{m1_dir}/model.pt"
    m1_ckpt = f"{m1_dir}/checkpoint.pt"
    if not os.path.exists(m1_model):
        cmd = [
            "python", "experiments/train.py",
            "--config", "results/spwm_v0_12_4_train_m1/config.yaml",
            "--model-type", "ann_rate",
            "--mode", "train_m1",
            "--seed", str(s),
            "--epochs", "240",
            "--representation-checkpoint", rep_model,
            "--device", "cuda",
            "--output-dir", m1_dir,
        ]
        if os.path.exists(m1_ckpt):
            cmd.extend(["--resume", m1_ckpt])
        else:
            cmd.append("--from-scratch")
        run_cmd(cmd)
    else:
        print(f"Seed {s} train_m1 already complete: {m1_model}")

# Sync trained ANN-Rate directories into repo results/ for Part 2
!mkdir -p results/ann_ablation
!cp -r "{DRIVE_RESULTS_DIR}/ann_ablation/"* results/ann_ablation/

In [ ]:
# 5. Part 2: Multi-Horizon Open-Loop Rollout (N=400, H=100, t_start=4)
!python scripts/eval_table1_ssot.py
!cp -r results/comparative_metrics_v0_12_4.json "{DRIVE_RESULTS_DIR}/" 2>/dev/null || true
!cp -r results/baselines "{DRIVE_RESULTS_DIR}/" 2>/dev/null || true